# Feature Engineering — Home Credit

Improve regression accuracy for predicting **AMT_CREDIT** (loan amount) by engineering new features from `application_train.csv`, then compare **baseline** vs **engineered** feature sets using Linear Regression and Random Forest.

**No target leakage:** ratios use income, goods price, and annuity only — never `AMT_CREDIT`.

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

RANDOM_STATE = 42
DATA_PATH = r"C:\Users\hchin\OneDrive\Desktop\ml_project\Machine Learning End to End Project\csv\application_train.csv"

## 2. Load data

In [2]:
df = pd.read_csv(DATA_PATH)
print(f"Dataset shape: {df.shape}")
df.head()

Dataset shape: (307511, 123)


,SK_ID_CURR,application_date,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,2018-01-01,1,Cash loans,M,N,Y,0,202500.0,406597.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,2018-01-01,0,Cash loans,F,N,N,0,270000.0,1293502.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,2018-01-01,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,2018-01-01,0,Cash loans,F,N,Y,0,135000.0,312682.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,2018-01-01,0,Cash loans,M,N,Y,0,121500.0,513000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Feature engineering helpers

Create derived numeric features and one-hot encode selected categoricals. Sentinel `365243` in `DAYS_EMPLOYED` marks unemployed clients.

In [3]:
UNEMPLOYED_SENTINEL = 365243
DAYS_PER_YEAR = 365.25


def build_engineered_features(data: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with engineered columns (no AMT_CREDIT-derived ratios)."""
    out = data.copy()

    out["AGE_YEARS"] = -out["DAYS_BIRTH"] / DAYS_PER_YEAR

    out["EMPLOYED_YEARS"] = np.where(
        out["DAYS_EMPLOYED"] == UNEMPLOYED_SENTINEL,
        0.0,
        -out["DAYS_EMPLOYED"] / DAYS_PER_YEAR,
    )

    out["GOODS_INCOME_RATIO"] = out["AMT_GOODS_PRICE"] / out["AMT_INCOME_TOTAL"]
    out["ANNUITY_INCOME_RATIO"] = out["AMT_ANNUITY"] / out["AMT_INCOME_TOTAL"]
    out["GOODS_TO_ANNUITY_RATIO"] = out["AMT_GOODS_PRICE"] / out["AMT_ANNUITY"]

    ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
    out["EXT_SOURCE_MEAN"] = out[ext_cols].mean(axis=1, skipna=True)
    out["EXT_SOURCE_MIN"] = out[ext_cols].min(axis=1, skipna=True)
    out["EXT_SOURCE_MAX"] = out[ext_cols].max(axis=1, skipna=True)

    out["INCOME_PER_CHILD"] = out["AMT_INCOME_TOTAL"] / (out["CNT_CHILDREN"] + 1)

    cat_dummies = pd.get_dummies(
        out[["NAME_CONTRACT_TYPE", "CODE_GENDER"]],
        drop_first=True,
        dtype=float,
    )
    out = pd.concat([out, cat_dummies], axis=1)

    return out


target = "AMT_CREDIT"

baseline_feature_cols = [
    "AMT_INCOME_TOTAL",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "CNT_CHILDREN",
    "DAYS_BIRTH",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
]

raw_cols_for_engineering = [
    target,
    *baseline_feature_cols,
    "DAYS_EMPLOYED",
    "CNT_FAM_MEMBERS",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
    "REGION_POPULATION_RELATIVE",
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
]

work_df = build_engineered_features(df[raw_cols_for_engineering].copy())

engineered_new_cols = [
    "AGE_YEARS",
    "EMPLOYED_YEARS",
    "GOODS_INCOME_RATIO",
    "ANNUITY_INCOME_RATIO",
    "GOODS_TO_ANNUITY_RATIO",
    "EXT_SOURCE_MEAN",
    "EXT_SOURCE_MIN",
    "EXT_SOURCE_MAX",
    "INCOME_PER_CHILD",
    "CNT_FAM_MEMBERS",
    "DAYS_REGISTRATION",
    "DAYS_ID_PUBLISH",
    "REGION_POPULATION_RELATIVE",
]

dummy_cols = [c for c in work_df.columns if c.startswith("NAME_CONTRACT_TYPE_") or c.startswith("CODE_GENDER_")]
engineered_feature_cols = baseline_feature_cols + engineered_new_cols + dummy_cols

print(f"Baseline features ({len(baseline_feature_cols)}): {baseline_feature_cols}")
print(f"\nNew engineered features ({len(engineered_new_cols)}): {engineered_new_cols}")
print(f"\nOne-hot categorical columns ({len(dummy_cols)}): {dummy_cols}")
print(f"\nTotal engineered feature matrix columns: {len(engineered_feature_cols)}")

Baseline features (8): ['AMT_INCOME_TOTAL', 'AMT_ANNUITY', 'AMT_GOODS_PRICE', 'CNT_CHILDREN', 'DAYS_BIRTH', 'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']

New engineered features (13): ['AGE_YEARS', 'EMPLOYED_YEARS', 'GOODS_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'GOODS_TO_ANNUITY_RATIO', 'EXT_SOURCE_MEAN', 'EXT_SOURCE_MIN', 'EXT_SOURCE_MAX', 'INCOME_PER_CHILD', 'CNT_FAM_MEMBERS', 'DAYS_REGISTRATION', 'DAYS_ID_PUBLISH', 'REGION_POPULATION_RELATIVE']

One-hot categorical columns (3): ['NAME_CONTRACT_TYPE_Revolving loans', 'CODE_GENDER_M', 'CODE_GENDER_XNA']

Total engineered feature matrix columns: 24


## 4. Evaluation helper & train/test split

Baseline: `dropna()` on base features (same as prior notebooks). Engineered: `dropna()` on the full engineered matrix. Both use **80/20** split with `random_state=42`. We also align on common valid rows so comparisons use identical samples.

In [4]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    return r2, mae, rmse


def prepare_splits(feature_cols, label=""):
    subset = work_df[[target] + feature_cols].dropna()
    X = subset[feature_cols]
    y = subset[target]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=RANDOM_STATE
    )
    print(f"{label} rows after dropna: {len(subset):,}")
    print(f"  Train: {len(X_train):,} | Test: {len(X_test):,}")
    return X_train, X_test, y_train, y_test


# Common index: valid for both baseline and engineered feature sets
baseline_valid = work_df[[target] + baseline_feature_cols].dropna().index
engineered_valid = work_df[[target] + engineered_feature_cols].dropna().index
common_idx = baseline_valid.intersection(engineered_valid)

aligned_df = work_df.loc[common_idx].copy()
print(f"Rows valid for baseline only:  {len(baseline_valid):,}")
print(f"Rows valid for engineered:     {len(engineered_valid):,}")
print(f"Common rows (used for compare): {len(common_idx):,}")

y_all = aligned_df[target]
X_base_all = aligned_df[baseline_feature_cols]
X_eng_all = aligned_df[engineered_feature_cols]

idx_train, idx_test = train_test_split(
    aligned_df.index, test_size=0.2, random_state=RANDOM_STATE
)

X_base_train, X_base_test = X_base_all.loc[idx_train], X_base_all.loc[idx_test]
X_eng_train, X_eng_test = X_eng_all.loc[idx_train], X_eng_all.loc[idx_test]
y_train, y_test = y_all.loc[idx_train], y_all.loc[idx_test]

print(f"\nAligned train size: {len(idx_train):,}")
print(f"Aligned test size:  {len(idx_test):,}")

Rows valid for baseline only:  109,483
Rows valid for engineered:     109,483
Common rows (used for compare): 109,483

Aligned train size: 87,586
Aligned test size:  21,897


## 5. Baseline models

Train **Linear Regression** and **Random Forest** (`n_estimators=100`, `max_depth=10`) on baseline features.

In [5]:
baseline_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(
        n_estimators=100, max_depth=10, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

baseline_results = []
for name, model in baseline_models.items():
    model.fit(X_base_train, y_train)
    r2, mae, rmse = evaluate_model(model, X_base_test, y_test)
    baseline_results.append({
        "Model": name,
        "Feature Set": "Baseline",
        "R²": r2,
        "MAE": mae,
        "RMSE": rmse,
    })
    print(f"{name} (Baseline):")
    print(f"  R²:   {r2:.4f}")
    print(f"  MAE:  {mae:,.2f}")
    print(f"  RMSE: {rmse:,.2f}")
    print()

Linear Regression (Baseline):
  R²:   0.9745
  MAE:  50,491.57
  RMSE: 65,922.50

Random Forest (Baseline):
  R²:   0.9872
  MAE:  29,503.57
  RMSE: 46,650.90



## 6. Engineered-feature models

Same models and hyperparameters on the expanded feature matrix.

In [6]:
engineered_results = []
fitted_eng_models = {}

for name, model in baseline_models.items():
    model.fit(X_eng_train, y_train)
    fitted_eng_models[name] = model
    r2, mae, rmse = evaluate_model(model, X_eng_test, y_test)
    engineered_results.append({
        "Model": name,
        "Feature Set": "Engineered",
        "R²": r2,
        "MAE": mae,
        "RMSE": rmse,
    })
    print(f"{name} (Engineered):")
    print(f"  R²:   {r2:.4f}")
    print(f"  MAE:  {mae:,.2f}")
    print(f"  RMSE: {rmse:,.2f}")
    print()

Linear Regression (Engineered):
  R²:   0.9759
  MAE:  47,977.85
  RMSE: 64,043.76

Random Forest (Engineered):
  R²:   0.9889
  MAE:  27,597.82
  RMSE: 43,490.34



## 7. Optional: tune Random Forest on engineered features

Small `RandomizedSearchCV` search to squeeze extra performance from tree-based model.

In [7]:
rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions={
        "n_estimators": [100, 150, 200],
        "max_depth": [10, 15, 20, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
    },
    n_iter=12,
    cv=3,
    scoring="neg_mean_squared_error",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

print("Tuning Random Forest on engineered features...")
rf_search.fit(X_eng_train, y_train)
print(f"Best params: {rf_search.best_params_}")

best_rf = rf_search.best_estimator_
r2, mae, rmse = evaluate_model(best_rf, X_eng_test, y_test)
tuned_rf_result = {
    "Model": "Random Forest (Tuned)",
    "Feature Set": "Engineered",
    "R²": r2,
    "MAE": mae,
    "RMSE": rmse,
}

print(f"\nRandom Forest (Tuned, Engineered):")
print(f"  R²:   {r2:.4f}")
print(f"  MAE:  {mae:,.2f}")
print(f"  RMSE: {rmse:,.2f}")

Tuning Random Forest on engineered features...
Best params: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_depth': None}

Random Forest (Tuned, Engineered):
  R²:   0.9959
  MAE:  12,766.90
  RMSE: 26,348.21


## 8. Final comparison table

In [8]:
all_results = baseline_results + engineered_results + [tuned_rf_result]
comparison_df = pd.DataFrame(all_results).sort_values("R²", ascending=False).reset_index(drop=True)

display_df = comparison_df.copy()
display_df["R²"] = display_df["R²"].map(lambda x: f"{x:.4f}")
display_df["MAE"] = display_df["MAE"].map(lambda x: f"{x:,.2f}")
display_df["RMSE"] = display_df["RMSE"].map(lambda x: f"{x:,.2f}")

print("Model | Feature Set | R² | MAE | RMSE")
print("=" * 70)
print(display_df.to_string(index=False))

comparison_df

Model | Feature Set | R² | MAE | RMSE
                Model Feature Set     R²       MAE      RMSE
Random Forest (Tuned)  Engineered 0.9959 12,766.90 26,348.21
        Random Forest  Engineered 0.9889 27,597.82 43,490.34
        Random Forest    Baseline 0.9872 29,503.57 46,650.90
    Linear Regression  Engineered 0.9759 47,977.85 64,043.76
    Linear Regression    Baseline 0.9745 50,491.57 65,922.50


,Model,Feature Set,R²,MAE,RMSE
0,Random Forest (Tuned),Engineered,0.995929,12766.900390,26348.213156
1,Random Forest,Engineered,0.988909,27597.819615,43490.344229
2,Random Forest,Baseline,0.987238,29503.571826,46650.903249
3,Linear Regression,Engineered,0.975948,47977.850126,64043.764870
4,Linear Regression,Baseline,0.974516,50491.569910,65922.500818


## 9. Improvement summary

In [9]:
def get_metric(results, model_name, feature_set, metric):
    row = next(r for r in results if r["Model"] == model_name and r["Feature Set"] == feature_set)
    return row[metric]


for model_name in ["Linear Regression", "Random Forest"]:
    base_r2 = get_metric(baseline_results, model_name, "Baseline", "R²")
    eng_r2 = get_metric(engineered_results, model_name, "Engineered", "R²")
    base_rmse = get_metric(baseline_results, model_name, "Baseline", "RMSE")
    eng_rmse = get_metric(engineered_results, model_name, "Engineered", "RMSE")
    print(f"{model_name}:")
    print(f"  Δ R²:   {eng_r2 - base_r2:+.4f}")
    print(f"  Δ RMSE: {eng_rmse - base_rmse:+,.2f}")
    print()

best_row = comparison_df.iloc[0]
print("Best overall result:")
print(f"  {best_row['Model']} | {best_row['Feature Set']}")
print(f"  R²:   {best_row['R²']:.4f}")
print(f"  MAE:  {best_row['MAE']:,.2f}")
print(f"  RMSE: {best_row['RMSE']:,.2f}")

Linear Regression:
  Δ R²:   +0.0014
  Δ RMSE: -1,878.74

Random Forest:
  Δ R²:   +0.0017
  Δ RMSE: -3,160.56

Best overall result:
  Random Forest (Tuned) | Engineered
  R²:   0.9959
  MAE:  12,766.90
  RMSE: 26,348.21


## 10. Conclusion

See the comparison table and improvement summary above. Feature engineering adds interpretable ratios (income/annuity/goods), aggregated external scores, demographics, and categorical signals. If **Engineered** rows beat **Baseline** for the same model, the new features helped; the Δ R² and Δ RMSE cells quantify the gain.

Random Forest typically benefits most from non-linear and interaction-rich features; Linear Regression may show smaller gains when original features already explain most variance (high baseline R²).